<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/dailychallenge_week9_day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installation
!pip install -q smolagents[transformers] wikipedia

# 1) KB
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))

# 2) Outils
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    name = "kb_lookup"
    description = "Looks up relevant information from a custom knowledge base."
    inputs = {"query": {"type": "string", "description": "The search query to match against KB entries."}}
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [f"[{item['source']}] {item['text']}" for item in self.kb if any(w in item["text"].lower() for w in q.split())]
        return "\n".join(matches) if matches else "No KB match."

class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {"type": "number", "description": "First number"},
        "b": {"type": "number", "description": "Second number"},
        "op": {"type": "string", "description": "Operation: 'add' or 'multiply'", "enum": ["add", "multiply"], "nullable": True}
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        return str(a * b) if op == "multiply" else str(a + b)

kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

# 3) Modèle - Utilisation de TinyLlama pour éviter les erreurs de logique de l'agent
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    model_id=MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
)
print("Model ready:", MODEL_ID)

# 4) Agent
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=3,
    instructions=(
        "You are an agentic AI that uses tools to answer questions. "
        "Use kb_lookup for KB queries and math_tool for arithmetic. "
        "If you lack evidence, say so and suggest a follow-up. "
        "Cite sources like [kb:source]."
    ),
)

# 5) Tests
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    try:
        result = agent.run(q)
        print("Answer:", result)
    except Exception as e:
        print(f"Error during run: {e}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.8 MB/s eta 0:00:00
KB entries: 5


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

[Step 3: Duration 7.32 seconds| Input tokens: 4,210 | Output tokens: 395]

[transformers] Both `max_new_tokens` (=4096) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 4: Duration 4.16 seconds| Input tokens: 4,917 | Output tokens: 507]

Answer: An agentic AI loop is a type of AI task that involves using tools to solve a problem repeatedly until a specific outcome is achieved. The loop is typically used in AI tasks that require a specific outcome to be achieved, such as solving a puzzle or completing a task. The AI agent uses tools to gather information, make decisions, and execute actions to achieve the desired outcome. The loop is repeated until the desired outcome is achieved, and the AI agent continues to use the same tools and actions to complete the task.


### Test Interactif
Utilisez cette cellule pour poser des questions personnalisées à votre agent.

In [ ]:
user_query = "Calcule 15 plus 27 et explique ce qu'est un outil dans la KB." # @param {type:"string"}

print(f"Test de la requête : {user_query}")
try:
    # On utilise .run() pour lancer le cycle de l'agent
    response = agent.run(user_query)
    print("\n--- RÉPONSE FINALE ---")
    print(response)
except Exception as e:
    print(f"Erreur lors de l'exécution : {e}")

Test de la requête : Calcule 15 plus 27 et explique ce qu'est un outil dans la KB.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Calcule 15 plus 27 et explique ce qu'est un outil dans la KB.                                                   │
│                                                                                                                 │
╰─ TransformersModel - TinyLlama/TinyLlama-1.1B-Chat-v1.0 ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] Both `max_new_tokens` (=4096) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error while parsing tool call from model output: The model output does not contain any JSON blob.

[Step 1: Duration 149.00 seconds| Input tokens: 1,227 | Output tokens: 4,096]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] Both `max_new_tokens` (=4096) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Step 2: Duration 48.26 seconds]

KeyboardInterrupt: 